In [1]:
import sys
from pathlib import Path
import pandas as pd

projectRoot = Path().resolve().parents[0]
print(f"path: {projectRoot}")
sys.path.append(str(projectRoot))

%load_ext autoreload
%autoreload 2

path: /workspace


In [2]:
# Prepare LLM batch inputs
!python /workspace/src/evaluation/benchmark_runner_batch.py prepare \
  /workspace/data/qa/llm_batch/qa_pairs.csv \
  /workspace/data/results/llm_batch/Qwen3-235B-A22B-Instruct-2507/requests.jsonl \
  /workspace/data/results/llm_batch/Qwen3-235B-A22B-Instruct-2507/mapping.csv \
  --provider nebius \
  --model Qwen/Qwen3-235B-A22B-Instruct-2507 \
  --temperature 0.0 \
  --max_tokens 200

Loaded 1562 QA pairs from /workspace/data/qa/llm_batch/qa_pairs.csv
Using provider: nebius
Prepared 4686 batch requests to /workspace/data/results/llm_batch/Qwen3-235B-A22B-Instruct-2507/requests.jsonl
Wrote mapping to /workspace/data/results/llm_batch/Qwen3-235B-A22B-Instruct-2507/mapping.csv


In [3]:
# Submit LLM batch job and store the output batch ID
from datetime import datetime
import subprocess

# Generate the file path for storing the batch ID
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
batch_id_file = projectRoot / f"BATCH_ID/{timestamp}_batchID"

# Run the command and capture the batch ID

result = subprocess.run(
    [
        "python", "/workspace/src/evaluation/benchmark_runner_batch.py", "submit",
        "/workspace/data/results/llm_batch/Qwen3-235B-A22B-Instruct-2507/requests.jsonl",
        "--provider", "nebius",
        "--window", "24h"
    ],
    stdout=subprocess.PIPE,
    text=True,
    check=True
)

# Extract the batch ID from the command output
batch_id = result.stdout.strip()

# Save the batch ID to the file
batch_id_file.parent.mkdir(parents=True, exist_ok=True)
with open(batch_id_file, "w") as file:
    file.write(batch_id)

print(f"Batch ID saved to: {batch_id_file}")

Batch ID saved to: /workspace/BATCH_ID/20260104_115746_batchID


In [8]:
# Check status of LLM batch job
# batch_id = "<batch_id>"  # Replace with the actual batch ID

# Read batch_id from the file
with open(batch_id_file, "r") as file:
    batch_id = file.read().strip()
batch_id = batch_id.split("id=")[1].split(",")[0]
    
!python /workspace/src/evaluation/benchmark_runner_batch.py status {batch_id} \
    --provider nebius

Batch batch_019b88de-f412-792a-9b65-54490f24cd9d status: completed
Counts: BatchRequestCounts(completed=4686, failed=0, total=4686, prompt_tokens=611047, completion_tokens=464755)
Output file id: 019b8984-a251-7f0e-9930-f66a6ce1a980


In [9]:
# Collect LLM batch results
!python /workspace/src/evaluation/benchmark_runner_batch.py collect \
  {batch_id} \
  /workspace/data/results/llm_batch/Qwen3-235B-A22B-Instruct-2507/output.jsonl \
  /workspace/data/results/llm_batch/Qwen3-235B-A22B-Instruct-2507/mapping.csv \
  /workspace/data/results/llm_batch/Qwen3-235B-A22B-Instruct-2507_results.csv \
  /workspace/data/results/llm_batch/Qwen3-235B-A22B-Instruct-2507_raw.csv \
  --model Qwen/Qwen3-235B-A22B-Instruct-2507 \
  --provider nebius

Saved batch output JSONL to /workspace/data/results/llm_batch/Qwen3-235B-A22B-Instruct-2507/output.jsonl
Wrote parsed results to /workspace/data/results/llm_batch/Qwen3-235B-A22B-Instruct-2507_results.csv (1562 rows)
Wrote raw outputs to /workspace/data/results/llm_batch/Qwen3-235B-A22B-Instruct-2507_raw.csv (1562 rows)
